# JAX Frontend (`pyepo.func.jax`)

This notebook trains a shortest-path predictor with the SPO+ loss in JAX/Flax, then swaps to the MPAX backend for a GPU-native, jitted training step. Pairs with the [JAX Frontend docs](https://khalil-research.github.io/PyEPO/build/html/content/examples/jax.html).

`pyepo.func.jax` mirrors `pyepo.func` so a JAX/Flax model can be trained end-to-end with `jax.grad`. Every loss keeps the same class name, constructor and call signature, and acronym alias as its PyTorch version — switching frameworks is a one-line import change:

```python
# torch:  from pyepo.func import SPOPlus
# jax:    from pyepo.func.jax import SPOPlus
```

It works with **any** PyEPO solver backend: MPAX is solved natively (GPU-native, jittable); every other backend (Gurobi, COPT, Pyomo, OR-Tools) is reached through `jax.pure_callback`.

## Install (Colab Only)

In [1]:
# loss frontend + MPAX fast path, and Flax + optax for the predictor/optimizer
!pip install pyepo[mpax]
!pip install flax optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill

## 1 Optimization Model and Data

A 5×5 grid shortest-path model and synthetic data — same scale as the PyTorch tutorials: 1 000 train / 1 000 test, `seed=42`, degree-4 polynomial, noise 0.5. The training split is converted to JAX arrays; the test split stays as numpy for regret evaluation.

In [2]:
import functools
import numpy as np
import jax
import jax.numpy as jnp
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import pyepo
from pyepo.data.dataset import optDataset

# 5x5 grid shortest path
grid = (5, 5)
num_feat = 5
deg = 4
e = 0.5
optmodel = pyepo.model.shortestPathModel(grid, backend="mpax")

# 1000 train + 1000 test (same split as notebook 03)
feats, costs = pyepo.data.shortestpath.genData(2000, num_feat, grid, deg, e, seed=42)
x_train, x_test, c_train, c_test = train_test_split(feats, costs, test_size=1000, random_state=42)

dataset_train = optDataset(optmodel, x_train, c_train)
dataset_test  = optDataset(optmodel, x_test,  c_test)
loader_train  = DataLoader(dataset_train, batch_size=32, shuffle=True,  drop_last=True)
loader_test   = DataLoader(dataset_test,  batch_size=32, shuffle=False)

# one sample as JAX array for model init shape
xj = jnp.asarray(x_train[:1], jnp.float32)
print('train:', len(dataset_train), '| test:', len(dataset_test), '| cost dim:', optmodel.num_cost)

train: 1000 | test: 1000 | cost dim: 40


## 2 The One-Line Switch

The only change from a PyTorch script is the import. The constructor kwargs (`reduction`, `processes`, `solve_ratio`, ...) and the call signature `spo(pred_cost, true_cost, true_sol, true_obj)` are identical.

In [3]:
# PyTorch:  from pyepo.func import SPOPlus
from pyepo.func.jax import SPOPlus

spo = SPOPlus(optmodel, reduction="mean")

## 3 End-to-End Training (Flax + optax)

A linear predictor maps features to per-edge costs; `jax.grad` differentiates the SPO+ loss through the (non-differentiable) solver via its `jax.custom_vjp` gradient rule, and optax applies the update.

In [4]:
import time
import optax
from flax import linen as nn

# linear predictor: features -> per-edge cost
predmodel = nn.Dense(optmodel.num_cost)
params = predmodel.init(jax.random.PRNGKey(0), xj)

optimizer = optax.adam(1e-2)
opt_state = optimizer.init(params)

elapsed = 0
for epoch in range(20):
    tick = time.time()
    for x_b, c_b, w_b, z_b in loader_train:
        xb = jnp.asarray(x_b)
        cb = jnp.asarray(c_b)
        wb = jnp.asarray(w_b)
        zb = jnp.asarray(z_b)
        loss, grads = jax.value_and_grad(
            lambda p: spo(predmodel.apply(p, xb), cb, wb, zb)
        )(params)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
    jax.block_until_ready(loss)
    tock = time.time()
    elapsed += tock - tick
    regret_val = pyepo.metric.regret(functools.partial(predmodel.apply, params), optmodel, loader_test)
    print("Epoch {:2},  Loss: {:9.4f},  Regret: {:7.4f}%".format(epoch + 1, loss.tolist(), regret_val * 100))
print("Total Elapsed Time: {:.2f} Sec.".format(elapsed))

/usr/local/lib/python3.12/dist-packages/pyepo/metric/regret.py:161: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  cp = torch.as_tensor(np.asarray(predmodel(x.numpy())), dtype=torch.float32)
MPAX solutions on cuda:0 differ from input device cpu; copying


Epoch  1,  Loss:    3.3601,  Regret: 24.1183%
Epoch  2,  Loss:    1.7993,  Regret:  8.9801%
Epoch  3,  Loss:    1.4074,  Regret:  8.1860%
Epoch  4,  Loss:    1.4667,  Regret:  7.6109%
Epoch  5,  Loss:    0.8723,  Regret:  7.1833%
Epoch  6,  Loss:    0.8994,  Regret:  7.5652%
Epoch  7,  Loss:    1.0835,  Regret:  7.5531%
Epoch  8,  Loss:    0.8833,  Regret:  7.3738%
Epoch  9,  Loss:    0.9729,  Regret:  7.2038%
Epoch 10,  Loss:    1.5482,  Regret:  7.5020%
Epoch 11,  Loss:    1.9120,  Regret:  7.4797%
Epoch 12,  Loss:    1.0615,  Regret:  7.0625%
Epoch 13,  Loss:    1.3598,  Regret:  7.6384%
Epoch 14,  Loss:    1.1471,  Regret:  7.2547%
Epoch 15,  Loss:    1.2561,  Regret:  7.5951%
Epoch 16,  Loss:    2.1858,  Regret:  7.4590%
Epoch 17,  Loss:    1.3426,  Regret:  7.6178%
Epoch 18,  Loss:    1.5543,  Regret:  7.4650%
Epoch 19,  Loss:    1.3618,  Regret:  8.1515%
Epoch 20,  Loss:    0.9607,  Regret:  7.4919%
Total Elapsed Time: 90.90 Sec.


## 4 Decision Quality Evaluation

`pyepo.metric.regret` now accepts any callable `f(x_numpy) → cost_array`, so a Flax model plugs in directly via `functools.partial(model.apply, params)` — no wrapper needed.

In [5]:
regret_val = pyepo.metric.regret(functools.partial(predmodel.apply, params), optmodel, loader_test)
print("Regret: {:.4f}%".format(regret_val * 100))

Regret: 7.5168%


## 5 GPU with MPAX + `jax.jit`

Swap the solver backend to MPAX (no other change) and JIT the whole training step by closing over the model. MPAX batch-solves on GPU, so there is no per-step CPU round-trip. The labels (`cj`/`wj`/`zj`) are reused; only the per-step solve moves to MPAX.

In [6]:
from pyepo.model.mpax.shortestpath import shortestPathModel as mpaxShortestPathModel

mpax_model = mpaxShortestPathModel(grid)
spo_mpax = SPOPlus(mpax_model, reduction="mean")

params = predmodel.init(jax.random.PRNGKey(0), xj)
opt_state = optimizer.init(params)

@jax.jit
def step(params, opt_state, xb, cb, wb, zb):
    loss, grads = jax.value_and_grad(
        lambda p: spo_mpax(predmodel.apply(p, xb), cb, wb, zb)
    )(params)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    return optax.apply_updates(params, updates), new_opt_state, loss

elapsed = 0
for epoch in range(20):
    tick = time.time()
    for x_b, c_b, w_b, z_b in loader_train:
        xb = jnp.asarray(x_b)
        cb = jnp.asarray(c_b)
        wb = jnp.asarray(w_b)
        zb = jnp.asarray(z_b)
        params, opt_state, loss = step(params, opt_state, xb, cb, wb, zb)
    jax.block_until_ready(loss)
    tock = time.time()
    elapsed += tock - tick
    regret_val = pyepo.metric.regret(functools.partial(predmodel.apply, params), optmodel, loader_test)
    print("Epoch {:2},  Loss: {:9.4f},  Regret: {:7.4f}%".format(epoch + 1, loss.tolist(), regret_val * 100))
print("Total Elapsed Time: {:.2f} Sec.".format(elapsed))

Epoch  1,  Loss:    4.1548,  Regret: 23.0273%
Epoch  2,  Loss:    1.9957,  Regret:  9.4541%
Epoch  3,  Loss:    1.7971,  Regret:  8.1735%
Epoch  4,  Loss:    1.1612,  Regret:  7.5295%
Epoch  5,  Loss:    1.2369,  Regret:  7.2181%
Epoch  6,  Loss:    0.9266,  Regret:  7.3422%
Epoch  7,  Loss:    1.4747,  Regret:  7.3009%
Epoch  8,  Loss:    1.1904,  Regret:  7.8966%
Epoch  9,  Loss:    1.1791,  Regret:  7.0812%
Epoch 10,  Loss:    1.0956,  Regret:  7.1916%
Epoch 11,  Loss:    1.3847,  Regret:  7.1643%
Epoch 12,  Loss:    1.4186,  Regret:  7.2727%
Epoch 13,  Loss:    1.0373,  Regret:  7.4670%
Epoch 14,  Loss:    0.9729,  Regret:  7.3815%
Epoch 15,  Loss:    0.9975,  Regret:  7.3990%
Epoch 16,  Loss:    1.1777,  Regret:  7.8552%
Epoch 17,  Loss:    0.8799,  Regret:  7.9321%
Epoch 18,  Loss:    1.2978,  Regret:  7.6561%
Epoch 19,  Loss:    1.2866,  Regret:  7.5614%
Epoch 20,  Loss:    1.0083,  Regret:  7.3673%
Total Elapsed Time: 56.43 Sec.


## Summary

- Every `pyepo.func` loss is ported — SPO+, PG, DBB/NID, DPO/DPOMul, PFY/PFYMul, I-MLE/AI-MLE, RFWO/RFY, listwise/pairwise/pointwise LTR, NCE/CMAP, CaVE — with the same class names, constructor kwargs, and acronym aliases, so switching frameworks is a one-line import change.
- Any solver backend works: MPAX natively (GPU-native, jittable), every other backend through `jax.pure_callback`.
- **`jax.jit`**: jit the whole training step by closing over the model (as above). The randomized losses (perturbed family + `implicitMLE`) are jittable when you pass an explicit `key`; `adaptiveImplicitMLE` is eager-only.
- **Caching / online pool growth** (`solve_ratio < 1`; the contrastive and ranking losses) are supported and faithful to PyTorch, but eager-only.
- **API**: every loss keeps PyTorch's signature, except `implicitMLE`/`adaptiveImplicitMLE`, which take `kappa`/`n_iterations`/`seed` scalars instead of a PyTorch `distribution` object.
- **Next:** the full loss reference in the [JAX Frontend docs](https://khalil-research.github.io/PyEPO/build/html/content/examples/jax.html).